## Image Analysis with Azure AI Content Understanding

This notebook uses **Azure AI Content Understanding** (the `azure-ai-contentunderstanding`
SDK) — one of Microsoft's recommended replacements for the deprecated Computer Vision
Image Analysis API. Content Understanding is a multimodal service that extracts structured
content from documents, images, audio, and video.

We use two prebuilt analyzers:

- **`prebuilt-imageSearch`** — returns a one-paragraph description (markdown + a `Summary`
  field) of a standalone image.
- **`prebuilt-read`** — performs OCR to read printed and handwritten text from an image.

> **Prerequisites:** A Microsoft Foundry resource in a region that supports Content
> Understanding. The `prebuilt-imageSearch` analyzer requires model deployments
> (`gpt-4.1-mini` and `text-embedding-3-large`) and a one-time default-model configuration
> on the resource. You also need the `Cognitive Services User` role on the resource.


### Installing Dependencies and Libraries

In [ ]:
%pip install azure-ai-contentunderstanding azure-identity pillow python-dotenv requests

### Setting Up the Environment

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
# Endpoint looks like: https://<your-resource-name>.services.ai.azure.com/
CONTENTUNDERSTANDING_ENDPOINT = os.getenv("CONTENTUNDERSTANDING_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv('CONTENTUNDERSTANDING_ENDPOINT')
TEXT_EMBEDDING_DEPLOYMENT = os.getenv('TEXT_EMBEDDING_DEPLOYMENT')

### Creating our Content Understanding Client

In [2]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential

# Authenticate with Entra ID (run `az login` first)
credential = DefaultAzureCredential()

client = ContentUnderstandingClient(endpoint=CONTENTUNDERSTANDING_ENDPOINT, credential=credential)

### Configure Default Model Deployments (one-time)

Prebuilt analyzers need to know which of your Microsoft Foundry model deployments to use.
Run this once per resource (or whenever your deployment names change). `prebuilt-imageSearch`
requires both a **completion** model (`gpt-4.1-mini`) and an **embedding** model
(`text-embedding-3-large`), so make sure both are deployed first.

In [3]:
model_deployments = {
    # prebuilt-*Search analyzers (e.g. prebuilt-imageSearch) use the mini completion alias
    "prebuilt-analyzer-completion-mini": AZURE_OPENAI_DEPLOYMENT,
    # ...plus the embedding alias for semantic search
    "prebuilt-analyzer-embedding": TEXT_EMBEDDING_DEPLOYMENT,
    # most other prebuilt analyzers use the standard completion alias
    "prebuilt-analyzer-completion": AZURE_OPENAI_DEPLOYMENT,
}

updated = client.update_defaults(model_deployments=model_deployments)

print("Configured model deployments:")
for model_name, deployment_name in (updated.model_deployments or {}).items():
    print(f"  {model_name}: {deployment_name}")

ServiceRequestError: HTTPSConnection(host='udemo-103-resource.services.azure.com', port=443): Failed to resolve 'udemo-103-resource.services.azure.com' ([Errno 11001] getaddrinfo failed)

### Define the Images to Analyze

Content Understanding analyzes content with **prebuilt analyzers** instead of a feature
list or a custom schema. For standalone image description we use `prebuilt-imageSearch`.

In [9]:
from azure.ai.contentunderstanding.models import AnalysisInput, AnalysisResult

images_to_analyze = [
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/building.jpg",
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/person.jpg",
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/street.jpg"
]

### Get Image Analysis Results

In [11]:
for image_url in images_to_analyze:
    print(f"\nAnalyzing image: {image_url}\n")

    # begin_analyze is a long-running operation; .result() waits for completion
    poller = client.begin_analyze(
        analyzer_id="prebuilt-imageSearch",
        inputs=[AnalysisInput(url=image_url)],
    )
    result: AnalysisResult = poller.result()
    content = result.contents[0]

    print("Description (markdown):")
    print(content.markdown)

    # prebuilt-imageSearch returns a one-paragraph Summary field
    summary = content.fields.get("Summary") if content.fields else None
    if summary is not None and getattr(summary, "value", None):
        print("\nSummary:")
        print(f"  {summary.value}")


Analyzing image: https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/building.jpg

Description (markdown):
![image](pages/1)


Summary:
  The image shows the United States Capitol building in Washington, D.C., with its iconic dome prominently centered. The building is surrounded by green trees and a large grassy lawn in the foreground. The sky above is partly cloudy, and a few people are walking on the lawn, adding a sense of scale to the scene.

Analyzing image: https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/person.jpg

Description (markdown):
![image](pages/1)


Summary:
  The image shows a man wearing glasses, a gray blazer, and a checkered shirt, standing and gesturing with his hands. He appears to be speaking or presenting. In the background, there is a large Microsoft logo consisting of four colored squares (red, green, blue, and yellow) on a dark backdrop.

Analyzing i

### Read Text from an Image with the prebuilt-read Analyzer

The `prebuilt-read` analyzer performs OCR and returns the recognized text as markdown.
It handles both printed and handwritten text.

![handwritten.jpg](./Images/handwritten.png)

In [ ]:
from PIL import Image
from io import BytesIO
import requests

ocr_image_url = "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/handwritten.png"

# Download the image so we can display it alongside the transcription
response = requests.get(ocr_image_url)
original_image = Image.open(BytesIO(response.content)).convert("RGB")

# Run OCR with the prebuilt-read analyzer
poller = client.begin_analyze(
    analyzer_id="prebuilt-read",
    inputs=[AnalysisInput(url=ocr_image_url)],
)
result: AnalysisResult = poller.result()

content = result.contents[0]

print("Transcribed text:\n")
print(content.markdown)


Transcribed text:

Hello,

Simply Noted has developed
incredible proprietary robotic
technology to write your message
and envelopes with a genuine
real pen. It is completely
indistinguishable from a humans
handwriting.

Try us today!
Simply Noted

